In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

from load_data import *
from aligner import *
from features.windowing import create_windows


[*********************100%***********************]  5 of 5 completed


# 2. Validate

In [2]:
# Missing values
data.isna().sum()

Price   Ticker
Close   GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
High    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Low     GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Open    GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
Volume  GLD       1226
        QQQ          0
        SPY          0
        TLT        644
        VNQ       1190
dtype: int64

In [3]:
data.index.duplicated().sum()
print(data.shape)
print(data.head())
print(data.columns)

(6539, 25)
Price      Close                               High                            \
Ticker       GLD        QQQ        SPY TLT VNQ  GLD        QQQ        SPY TLT   
Date                                                                            
2000-01-03   NaN  79.839661  91.132751 NaN NaN  NaN  81.050948  92.895095 NaN   
2000-01-04   NaN  74.362541  87.568939 NaN NaN  NaN  78.786375  90.271201 NaN   
2000-01-05   NaN  72.466614  87.725540 NaN NaN  NaN  75.521166  88.685038 NaN   
2000-01-06   NaN  67.489807  86.315697 NaN NaN  NaN  74.151891  88.665489 NaN   
2000-01-07   NaN  75.837143  91.328590 NaN NaN  NaN  75.837143  91.328590 NaN   

Price           ... Open                               Volume            \
Ticker     VNQ  ...  GLD        QQQ        SPY TLT VNQ    GLD       QQQ   
Date            ...                                                       
2000-01-03 NaN  ...  NaN  81.050948  92.895095 NaN NaN    NaN  36345200   
2000-01-04 NaN  ...  NaN  77.522423  89.

In [4]:
close_prices = data["Close"]

for ticker in close_prices.columns:
    print(
        ticker,
        close_prices[ticker].first_valid_index(),
        close_prices[ticker].last_valid_index()
    )

GLD 2004-11-18 00:00:00 2025-12-31 00:00:00
QQQ 2000-01-03 00:00:00 2025-12-31 00:00:00
SPY 2000-01-03 00:00:00 2025-12-31 00:00:00
TLT 2002-07-30 00:00:00 2025-12-31 00:00:00
VNQ 2004-09-29 00:00:00 2025-12-31 00:00:00


# 3. Align assets

In [5]:
# We need to align so all 4 ETF:s start at the same time to match future matrix
# For the moment, we work with only one column
close_prices = data["Close"]

aligned_prices = align_assets(close_prices)
print(aligned_prices.head())
print(aligned_prices.shape)
aligned_prices.isna().sum().sum()

Ticker            GLD        QQQ        SPY        TLT        VNQ
Date                                                             
2004-11-18  44.380001  33.120090  79.745239  43.988651  21.189306
2004-11-19  44.779999  32.605877  78.858742  43.637611  20.970304
2004-11-22  44.950001  32.917759  79.234863  43.865047  21.115002
2004-11-23  44.750000  32.867195  79.355736  43.919460  21.204952
2004-11-24  45.049999  33.153801  79.543785  43.919460  21.572586
(5313, 5)


np.int64(0)

# 4. Convert raw to log returns

In [6]:
# Using r_t = log(P_t / P_{t-1})
log_returns = np.log(aligned_prices / aligned_prices.shift(1))

In [7]:
# Remove the first NaN raw
log_returns = log_returns.dropna()
print(log_returns.shape)
log_returns.describe()

(5312, 5)


Ticker,GLD,QQQ,SPY,TLT,VNQ
count,5312.000000,5312.000000,5312.000000,5312.000000,5312.000000
mean,0.000412,0.000549,0.000403,0.000125,0.000265
std,0.011132,0.013615,0.011983,0.009230,0.018122
min,-0.091905,-0.127592,-0.115887,-0.069010,-0.217083
25%,-0.005113,-0.005178,-0.003944,-0.005346,-0.006470
50%,0.000588,0.001146,0.000717,0.000445,0.000768
75%,0.006217,0.007321,0.005776,0.005505,0.007510
max,0.106974,0.114798,0.135577,0.072502,0.157059


# 5. Create rolling observation window

In [8]:
# Seeing only today's return is not relevant enough
# We also care to see how turbulent has the asset been recently
# we compute the std_dev or returns σ_t=std(r_t−19,...,r_t), 20 is approx. 1 month of trades
volatility_20 = log_returns.rolling(20).std()
print(volatility_20.head(25)) # volatility = 0.009785 ==> 0.97%

Ticker           GLD       QQQ       SPY       TLT       VNQ
Date                                                        
2004-11-19       NaN       NaN       NaN       NaN       NaN
2004-11-22       NaN       NaN       NaN       NaN       NaN
2004-11-23       NaN       NaN       NaN       NaN       NaN
2004-11-24       NaN       NaN       NaN       NaN       NaN
2004-11-26       NaN       NaN       NaN       NaN       NaN
2004-11-29       NaN       NaN       NaN       NaN       NaN
2004-11-30       NaN       NaN       NaN       NaN       NaN
2004-12-01       NaN       NaN       NaN       NaN       NaN
2004-12-02       NaN       NaN       NaN       NaN       NaN
2004-12-03       NaN       NaN       NaN       NaN       NaN
2004-12-06       NaN       NaN       NaN       NaN       NaN
2004-12-07       NaN       NaN       NaN       NaN       NaN
2004-12-08       NaN       NaN       NaN       NaN       NaN
2004-12-09       NaN       NaN       NaN       NaN       NaN
2004-12-10       NaN    

In [9]:
# We add momentum to observe the general direction, up or down
# (momentum is supposed to be smaller than volatility, because avg_return << std_dev)
momentum_20 = log_returns.rolling(20).mean()
print(momentum_20.head(25))

Ticker           GLD       QQQ       SPY       TLT       VNQ
Date                                                        
2004-11-19       NaN       NaN       NaN       NaN       NaN
2004-11-22       NaN       NaN       NaN       NaN       NaN
2004-11-23       NaN       NaN       NaN       NaN       NaN
2004-11-24       NaN       NaN       NaN       NaN       NaN
2004-11-26       NaN       NaN       NaN       NaN       NaN
2004-11-29       NaN       NaN       NaN       NaN       NaN
2004-11-30       NaN       NaN       NaN       NaN       NaN
2004-12-01       NaN       NaN       NaN       NaN       NaN
2004-12-02       NaN       NaN       NaN       NaN       NaN
2004-12-03       NaN       NaN       NaN       NaN       NaN
2004-12-06       NaN       NaN       NaN       NaN       NaN
2004-12-07       NaN       NaN       NaN       NaN       NaN
2004-12-08       NaN       NaN       NaN       NaN       NaN
2004-12-09       NaN       NaN       NaN       NaN       NaN
2004-12-10       NaN    

# 6. Generate derived features

In [10]:
# Combine the features
features = pd.concat(
    [
        log_returns,
        volatility_20,
        momentum_20
    ],
    axis=1,
    keys=["ret", "vol", "mom"]
)
# Drop the first 19 NaN
features = features.dropna()

print(features.shape)
print(features.head())

(5293, 15)
                 ret                                               vol  \
Ticker           GLD       QQQ       SPY       TLT       VNQ       GLD   
Date                                                                     
2004-12-17  0.011608 -0.002556 -0.006692 -0.001352  0.011262  0.009785   
2004-12-20  0.003389 -0.006864  0.000251  0.001802 -0.003506  0.009587   
2004-12-21 -0.002710  0.012171  0.007672  0.003031  0.011349  0.009544   
2004-12-22 -0.004533  0.001260  0.002406 -0.003370  0.007429  0.009546   
2004-12-23  0.005663  0.000754  0.000746 -0.003267 -0.007606  0.009506   

                                                         mom            \
Ticker           QQQ       SPY       TLT       VNQ       GLD       QQQ   
Date                                                                     
2004-12-17  0.008715  0.005472  0.007788  0.008975 -0.000215  0.000705   
2004-12-20  0.008043  0.004732  0.007557  0.008570 -0.000494  0.001144   
2004-12-21  0.008209  0.00

# 7. Create Splits

In [11]:
# Train, Validation, Test
train_features = features.loc[:'2018-12-31']
val_features = features.loc['2019-01-01':'2021-12-31']
test_features = features.loc['2022-01-01':]

print("Train:", train_features.shape)
print("Validation:", val_features.shape)
print("Test:", test_features.shape)

print(train_features.index.min(), train_features.index.max())
print(val_features.index.min(), val_features.index.max())
print(test_features.index.min(), test_features.index.max())

Train: (3533, 15)
Validation: (757, 15)
Test: (1003, 15)
2004-12-17 00:00:00 2018-12-31 00:00:00
2019-01-02 00:00:00 2021-12-31 00:00:00
2022-01-03 00:00:00 2025-12-31 00:00:00


# 8. Normalize

In [12]:
# Use z-score standardization
scaler = StandardScaler()
scaler.fit(train_features)

# Transform each split
train_scaled = pd.DataFrame(
    scaler.transform(train_features),
    index=train_features.index,
    columns=train_features.columns,
)

val_scaled = pd.DataFrame(
    scaler.transform(val_features),
    index=val_features.index,
    columns=val_features.columns,
)

test_scaled = pd.DataFrame(
    scaler.transform(test_features),
    index=test_features.index,
    columns=test_features.columns,
)

train_scaled.describe() # mean is approx 0 and std_dev approx 1

ret                                                          \
Ticker           GLD           QQQ           SPY           TLT           VNQ   
count   3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03   
mean    1.206696e-17 -5.027899e-18  7.039059e-18 -4.022319e-18 -1.231835e-17   
std     1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00   
min    -7.887357e+00 -7.404387e+00 -8.844775e+00 -6.037034e+00 -1.105058e+01   
25%    -4.762151e-01 -4.158714e-01 -3.477944e-01 -5.826283e-01 -3.491559e-01   
50%     1.779632e-02  4.003541e-02  2.986088e-02  2.890010e-02  2.250161e-02   
75%     5.173026e-01  4.879840e-01  4.319103e-01  5.894609e-01  3.721321e-01   
max     9.127083e+00  8.986748e+00  1.151427e+01  5.822602e+00  7.971886e+00   

                 vol                                                          \
Ticker           GLD           QQQ           SPY           TLT           VNQ   
count   3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03   
mean   -1.287142e-16  1.287142e-16 -1.287142e-16 -1.126249e-16 -3.217856e-17   
std     1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00   
min    -1.569890e+00 -1.311716e+00 -1.089188e+00 -1.794278e+00 -8.087353e-01   
25%    -7.046050e-01 -6.028588e-01 -5.877639e-01 -6.928347e-01 -5.174523e-01   
50%    -2.566559e-01 -2.720534e-01 -2.832146e-01 -2.657144e-01 -3.323127e-01   
75%     4.645889e-01  2.971564e-01  2.448722e-01  5.188413e-01  6.396837e-02   
max     5.882841e+00  6.554039e+00  7.164029e+00  4.757344e+00  6.690873e+00   

                 mom                                                          
Ticker           GLD           QQQ           SPY           TLT           VNQ  
count   3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03  3.533000e+03  
mean    3.217856e-17 -3.217856e-17  2.413392e-17  1.206696e-17  2.614508e-17  
std     1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00  1.000142e+00  
min    -4.445788e+00 -6.701138e+00 -8.169512e+00 -4.542930e+00 -8.102788e+00  
25%    -5.793645e-01 -4.791845e-01 -4.084880e-01 -6.320109e-01 -4.252625e-01  
50%    -1.929479e-02  1.210122e-01  1.538734e-01  1.086490e-02  1.026415e-01  
75%     6.326394e-01  6.289555e-01  5.590670e-01  5.798448e-01  5.178196e-01  
max     3.992733e+00  4.328688e+00  4.753387e+00  5.881362e+00  6.018177e+00

In [13]:
# Convert time series to window observation
WINDOW_SIZE = 30

train_windows, train_dates = create_windows(
    train_scaled,
    WINDOW_SIZE,
)
val_windows, val_dates = create_windows(
    val_scaled,
    WINDOW_SIZE,
)
test_windows, test_dates = create_windows(
    test_scaled,
    WINDOW_SIZE,
)